### **01. What an endpoint is?**

**An endpoint is just a URL path on your backend that does a specific job. In your code, / is one endpoint, /health is another, and /chat is the main one.**

Example:

1. GET / means “go to the home path of the API.”
2. GET /health means “check if the backend is alive.”
3. POST /chat means “send a chat message to the backend.”

*So if your backend runs on http://localhost:8000, then:*
* http://localhost:8000/
* http://localhost:8000/health
* http://localhost:8000/chat

are three different doors into the same server.

### **02. Common Imports in FastAPI**

In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from openai import OpenAI
import os
from dotenv import load_dotenv
from typing import Optional
import uuid

1. **FastAPI** creates the backend app.
2. **HTTPException** lets you return an error properly if something breaks.
3. **CORSMiddleware** allows your frontend on port 3000 to talk to your backend on port 8000.
4. **BaseModel** helps define the shape of incoming and outgoing JSON data.
5. **OpenAI** is the client used to call the OpenAI API.
6. **os** reads environment variables like your API key and allowed origins.
7. **load_dotenv** loads values from .env into your program.
8. **Optional** means a value may or may not be present.
9. **uuid** helps create a unique session ID.

### **03. Create the app**

In [2]:
app = FastAPI()

**app = your backend program**

### **04. CORS setup**

In [3]:
origins = os.getenv("CORS_ORIGINS", "http://localhost:3000").split(",")
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

**Frontend and backend are running on different addresses:**
* frontend: http://localhost:3000
* backend: http://localhost:8000

*Browsers treat those as different origins, so they block requests unless the backend explicitly says, “yes, I allow that frontend.” That permission layer is what CORS is doing here.*

**Line by line:**
1. os.getenv("CORS_ORIGINS", "http://localhost:3000") reads the allowed origin from .env, and if it is missing, it defaults to http://localhost:3000.
2. .split(",") turns a comma-separated string into a Python list, so later you could allow multiple frontend origins if needed.
3. app.add_middleware(...) adds CORS rules to the app.
4. allow_origins=origins says which frontend URLs are allowed.
5. allow_credentials=True allows cookies/auth headers if ever needed.
6. allow_methods=["*"] means all HTTP methods are allowed, like GET and POST.
7. allow_headers=["*"] means all headers are allowed.

### **05. OpenAI client**

In [4]:
client = OpenAI()

**This creates the object that talks to OpenAI’s API. Because we already loaded .env, the client can pick up OPENAI_API_KEY from the environment.**

Think of client as “phone” to OpenAI:
* backend uses the phone,
* sends the user’s message,
* gets the model’s reply back.

### **06. Personality loader**

In [8]:
def load_personality():
    with open("/mnt/d/study_2/TWIN/backend/me.txt", "r", encoding="utf-8") as f:
        return f.read().strip()
    
PERSONALITY = load_personality()

*This part reads me.txt file.*

In [9]:
PERSONALITY

'You are the digital twin of Prateek Ghorawat, M.Eng in AI Product Development, an AI/ML engineer and data professional based in the Munich area who has worked on Process AI at BMW and product analytics at Celonis. Speak and reason as Prateek: explain technical topics in plain language, prefer Python-first solutions, and cite your practical experience in data pipelines, ML systems, RAG/LLM tooling, and dashboarding when relevant. Be helpful, concise, and professional; if you don’t know an exact fact, say you’re unsure and offer how to find or verify it.\n\nA few optional lines you can append (pick any you like)\n\nPreferred phrasing: keep answers short, actionable, and Python-focused.\n\nPersonality notes: calm, confident, patient, and slightly nerdy (likes clean code and clear diagrams).\n\nSafety boundary: never reveal personal contact details beyond the ones on Prateek’s public CV or site.'

### **07. Pydantic**

Think of Pydantic as a smart form checker. Tell it, “I expect data shaped like this,” and it checks whether the incoming data matches that shape before your code uses it.

In [10]:
class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None
    

*you are basically saying: “I expect an object with a message field that must be text, and a session_id field that can be text or missing.”*

#### **07.01 Why FastAPI uses it**

FastAPI uses Pydantic to define the contract for API input and output. That means it can automatically validate incoming request data, parse it into Python objects, and also structure outgoing responses in a predictable way.

So instead of manually checking things like:

* does this JSON have message?
* is message a string?
* is session_id missing or a string?

**FastAPI + Pydantic handle that.**

#### **07.02 BaseModel**

**BaseModel** is just the parent class you use to create a data model. Pydantic docs and guides describe it as the core building block for defining structured, validated data using Python type hints.

*So in the Function above*
* ChatRequest is the name o model.
* message: str means message is required and must be a string.
* session_id: Optional[str] = None means session_id can be a string, but it is not required.

**Request model**

In [11]:
class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None

**Response model**

In [12]:
class ChatResponse(BaseModel):
    response: str
    session_id: str

### **08. Endpoints**

#### **08.01. Root endpoint**

* **GET /health means: “backend, tell me if you are alive.”**
* **POST /chat means: “backend, here is my message, please process it and reply.”**



In [14]:
@app.get("/")
async def root():
    return {"message": "AI Digital Twin API"}

Line by line:

* **@app.get("/")** = register a GET endpoint at /.
* **async def root():** = this is the Python function FastAPI will run.
* **return {"message": "AI Digital Twin API"}** = send JSON back to the client.

*If you open http://localhost:8000/ in the browser, the browser is making a GET request, so you get:*

**{"message":"AI Digital Twin API"}**

#### **08.02. Health endpoint**

In [15]:
@app.get("/health")
async def health_check():
    return {"status": "healthy"}

**It means: when someone makes a GET request to /health, return a simple status message. This is commonly used to check whether the backend is running properly.**

#### **08.03. Chat endpoint**

In [16]:
@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    pass

Line by Line:

* **@app.post("/chat") =** create a POST endpoint at /chat.
* **response_model=ChatResponse** = the output should follow the ChatResponse shape.
* **request: ChatRequest** = the incoming data should follow the ChatRequest shape.

**This is used because /chat is not just “show me a page.” It is “here is my message, now do work and return a reply.” POST is meant for sending data to the server.**

*The frontend sends JSON to /chat, for example:*

{

  "message": "Hello",

  "session_id": "abc123"
  
}

**FastAPI reads that JSON and converts it into a ChatRequest object because of this part:**

**request: ChatRequest**

So inside the function, you can do:
* **request.message**
* **request.session_id**

instead of manually digging through raw JSON.



In [17]:
@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    try:
        # Generate session ID if not provided
        session_id = request.session_id or str(uuid.uuid4())

        # Create system message with personality
        # NOTE: No memory - each request is independent!
        messages = [
            {"role": "system", "content": PERSONALITY},
            {"role": "user", "content": request.message},
        ]

        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini", 
            messages=messages
        )

        return ChatResponse(
            response=response.choices[0].message.content, 
            session_id=session_id
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

### **09. Run the App**

In [ ]:
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [19]:
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [142658]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [142658]
